# OOU-CSC 333 · Computer Programming III (Laboratory)
## Week 12: Integrated Project, Testing, Documentation and Review  ·  Starter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakinat-Folorunso/CSC333-Python-Lab/blob/main/notebooks/week12_integrated_project_and_review/CSC333_Week12_Starter.ipynb)

| | |
|---|---|
| **Instructor** | Dr Sakinat Folorunso, Associate Professor of AI Systems |
| **Department** | Department of Computer Science |
| **University** | Olabisi Onabanjo University, Ago-Iwoye, Nigeria |
| **Contact** | sakinat.folorunso@oouagoiwoye.edu.ng |
| **Session length** | 3 hours |

🟦 **STARTER notebook**: you will complete the activities, tasks and assignment yourself.

### 🎯 Learning outcomes
By the end of this lab you will be able to:

- Plan, build, test and document a small Python data application from a clear brief.
- Organise code into a **module**, a **test file** and a **README** that another person can run.
- Find and fix seeded defects under time pressure (mock practical), then explain the fix.
- Prepare and deliver a 5-minute demonstration that links design decisions to evidence.
- Reflect honestly on what you can now do, and what to practise next.

### 🧭 How to work through this notebook
1. Run the **Setup cell** first (Runtime → Run all is fine, or press `Shift + Enter` on each cell).
2. Read each markdown explanation, then read the **comment on every line** of the code.
3. Do the **🧠 Predict–Run–Explain** activities *before* you run the code: write your guess first.
4. Complete the **🛠️ Tasks**. Replace every `TODO`. Run the ✅ check cell under each task.
5. Finish the **📝 Assignment**, answer the **Quick check**, then complete the reflection and declaration at the end.


In [ ]:
# ===== SETUP CELL: run this first in every notebook =====
# check() is a small helper that tells you whether your answer is right.
# It never crashes your notebook: it only prints a friendly message.
def check(condition, success, hint):
    # condition : True or False, the result of a test you pass in
    # success   : message printed when the test passes
    # hint      : message printed when the test fails
    if condition:                       # if the test passed ...
        print("✅ Correct:", success)     # ... celebrate
    else:                               # otherwise ...
        print("❌ Not yet:", hint)        # ... show a hint and try again

print("Setup complete. Helper check() is ready.")  # confirm the cell ran

## 1. From exercises to a finished piece of work
Weeks 1 to 11 built the parts. This week you assemble them into **one reliable mini-application** and learn how professionals *deliver* it: code that runs, tests that prove it, and documentation that lets someone else use it.

### The project: Student Records Analyser (fictional data)
| Stage | What the program does | Weeks used |
|---|---|---|
| 1. Generate | Create a reproducible table of student records (`seed=42`) | 9, 10 |
| 2. Clean/validate | Reject impossible values, standardise labels | 3, 11 |
| 3. Store | Save and load records with **parameterised SQLite** queries | 11 |
| 4. Analyse | Grade bands, pass rate, best programme, top students | 4, 5, 10 |
| 5. Report | A chart and a printed summary | 11 |
| 6. Verify | Automated tests with `pytest` | 11 |
| 7. Document | A `README.md` anyone can follow | 0 |

### Marking guide for the final project (100 marks)
| Criterion | Marks | Excellent looks like |
|---|---|---|
| Correctness | 30 | Every function gives the right answer on normal, boundary and invalid input |
| Code quality | 20 | Clear names, every line understood, functions that do **one** job, no repeated code |
| Testing | 20 | Normal + boundary + invalid tests that fail when the code is broken |
| Documentation | 15 | README with install, usage, tests, limitations, author |
| Demonstration | 15 | 5 minutes, live run, honest about limitations |

> **Academic integrity.** You may discuss ideas, but the code you submit must be your own and you must be able to explain every line. Cite any help (a person, a tutorial, an AI tool) in your README. Being unable to explain your own submission is treated as a breach of integrity.

### The professional habit: *small, run, test, repeat*
Write a few lines, run them, test them, commit them. Do not write 200 lines and then debug. In the lab practical you will also be asked to **find a bug**: read the error, reproduce it with the smallest example, form a hypothesis, change one thing, test again.


In [ ]:
%%writefile records_app.py
"""records_app.py : the data layer of the Student Records Analyser (fictional data).
Author: Dr Sakinat Folorunso, Department of Computer Science, Olabisi Onabanjo University (teaching module).
"""
import sqlite3                                      # database (built into Python)
import numpy as np                                  # numerical arrays and random numbers
import pandas as pd                                 # tables

PROGRAMMES = ["CS", "DS", "CYB", "SE", "ICT"]       # the official programme codes

def make_records(n=30, seed=42):
    """Return a reproducible table of n fictional student records."""
    rng = np.random.default_rng(seed)               # seeded generator: same numbers every run
    return pd.DataFrame({
        "student_id": np.arange(1, n + 1),                              # 1, 2, ..., n
        "programme": rng.choice(PROGRAMMES, size=n),                    # random programme code
        "score": rng.integers(25, 100, size=n),                         # whole-number scores in [25, 99]
    })

def pass_rate(df, pass_mark=40):
    """Fraction of students with score >= pass_mark (a number between 0 and 1)."""
    return float((df["score"] >= pass_mark).mean())   # True counts as 1, so the mean is the fraction

def save_to_db(df, con):
    """Store a records table in SQLite using parameterised queries."""
    with con:                                       # one transaction
        con.execute("CREATE TABLE IF NOT EXISTS records (student_id INTEGER PRIMARY KEY, programme TEXT NOT NULL, score INTEGER)")
        rows = [(int(i), str(p), int(s)) for i, p, s in df.itertuples(index=False)]   # numpy ints -> plain Python ints
        con.executemany("INSERT OR REPLACE INTO records(student_id, programme, score) VALUES (?, ?, ?)", rows)

def load_from_db(con):
    """Return all stored records as a table, ordered by student_id."""
    return pd.read_sql_query("SELECT student_id, programme, score FROM records ORDER BY student_id", con)

In [ ]:
# --- DEMO 1: use the module like a user of your application would -----------------------------------------------
import sqlite3
import numpy as np                               # used later in the tasks
import pandas as pd
import records_app as ra                           # import the file we just wrote

df = ra.make_records()                             # stage 1: generate (seed 42 inside)
print(df.head())                                   # first five rows
print("rows:", len(df), "| pass rate:", round(ra.pass_rate(df), 3))

con = sqlite3.connect(":memory:")                  # stage 3: store
ra.save_to_db(df, con)
back = ra.load_from_db(con)
print("round trip identical:", back.equals(df))    # the data survives a trip through SQLite
con.close()

In [ ]:
%%writefile test_records_app.py
# test_records_app.py : the safety net for records_app.py (run with pytest)
import sqlite3
import records_app as ra

def test_make_records_is_reproducible():            # NORMAL: same seed gives same table
    assert ra.make_records(10, seed=42).equals(ra.make_records(10, seed=42))

def test_make_records_scores_in_range():            # BOUNDARY: scores must stay inside [25, 99]
    df = ra.make_records(200, seed=1)
    assert df["score"].min() >= 25 and df["score"].max() <= 99

def test_pass_rate_boundary():                      # BOUNDARY: exactly 40 passes, 39 does not
    df = ra.make_records(3)
    df["score"] = [40, 39, 100]
    assert abs(ra.pass_rate(df) - 2 / 3) < 1e-9

def test_database_round_trip():                     # NORMAL: save then load returns identical data
    con = sqlite3.connect(":memory:")
    df = ra.make_records(15)
    ra.save_to_db(df, con)
    assert ra.load_from_db(con).equals(df)
    con.close()

In [ ]:
# --- DEMO 2: run the whole test suite ---------------------------------------------------------------------------
import pytest
code = pytest.main(["-q", "test_records_app.py"])  # -q = quiet output
print("exit code:", int(code), "(0 means all tests passed)")

In [ ]:
# --- DEMO 3: a README is part of the program --------------------------------------------------------------------
readme_example = """# Student Records Analyser

## Purpose
Generates, cleans, stores and summarises fictional student records.

## Installation
Python 3.11+ and: pip install numpy pandas matplotlib pytest

## Usage
    import records_app as ra
    df = ra.make_records()
    print(ra.pass_rate(df))

## Testing
    pytest -q

## Limitations
Data are synthetic; scores are whole numbers from 25 to 99.

## Author
Your Name, Matric Number, Department of Computer Science, Olabisi Onabanjo University
"""
print(readme_example)

### 🧠 Class activity: Spot the bug before you run
**Predict → Run → Explain**

This function should return the fraction of scores that are **at least** 40. Predict the output for `[40, 39, 100]`, then run it.

**✍️ My prediction (write before running):** _write here_

In [ ]:
def buggy_pass_rate(scores):
    passed = 0
    for s in scores:
        if s > 40:                       # <- suspicious
            passed += 1
    return passed / len(scores)

print(buggy_pass_rate([40, 39, 100]))

**🔍 Explanation**

_Compare the output with your prediction. In your own words, explain any difference. Write your explanation here._

### 🧠 Class activity: Why seed_everything?
**Predict → Run → Explain**

Run the cell twice (use Runtime > Run all). Predict which line changes between runs and which never changes.

**✍️ My prediction (write before running):** _write here_

In [ ]:
import random
import numpy as np
print("unseeded :", random.random())
print("seeded   :", np.random.default_rng(42).integers(0, 100, size=3))

**🔍 Explanation**

_Compare the output with your prediction. In your own words, explain any difference. Write your explanation here._

## 🛠️ Tasks
These tasks complete the *analysis* layer of the application. The module `records_app` from the demos is already imported as `ra`.


### 🛠️ Task 1: Grade bands

Write `grade_band(score)` using these cut-offs: **A** 70 to 100, **B** 60 to 69, **C** 50 to 59, **D** 45 to 49, **E** 40 to 44, **F** 0 to 39. A score below 0, above 100, or not a number must raise `ValueError`.

In [ ]:
def grade_band(score):
    # TODO: validate first (number, 0..100), then return "A".."F" with if / elif
    pass

print(grade_band(72))

In [ ]:
# ✅ CHECK YOUR WORK: run this cell after you finish the task
try:                                   # run the checks safely
    cases = {100: "A", 70: "A", 69: "B", 60: "B", 59: "C", 50: "C", 49: "D", 45: "D", 44: "E", 40: "E", 39: "F", 0: "F"}
    wrong = {s: grade_band(s) for s, g in cases.items() if grade_band(s) != g}
    check(not wrong, "all 12 boundary scores are in the correct band", f"check these scores: {wrong}")
    def _raises(v):
        try:
            grade_band(v)
        except ValueError:
            return True
        return False
    check(all(_raises(v) for v in (-1, 101, "70", None)), "invalid inputs raise ValueError", "validate before comparing")
except Exception as err:               # an unfinished TODO can cause an error: that is normal
    print('❌ Not yet: your code raised', type(err).__name__, '-', err, '(finish the TODO above first)')

### 🛠️ Task 2: Summary statistics

Write `summarise(df)` returning a dictionary with keys `n` (rows), `mean` (mean score rounded to 2 dp), `median`, `pass_rate` (use `ra.pass_rate`, rounded to 3 dp) and `best_programme` (the programme with the highest **mean** score).

In [ ]:
def summarise(df):
    # TODO: build and return the dictionary
    pass

records = ra.make_records()
print(summarise(records))

In [ ]:
# ✅ CHECK YOUR WORK: run this cell after you finish the task
try:                                   # run the checks safely
    small = pd.DataFrame({"student_id": [1, 2, 3, 4], "programme": ["CS", "CS", "SE", "SE"], "score": [90, 30, 60, 70]})
    s = summarise(small)
    check(set(s) == {"n", "mean", "median", "pass_rate", "best_programme"}, "dictionary has the five keys", "use exactly these key names")
    check(s["n"] == 4 and s["mean"] == 62.5 and s["median"] == 65.0, "n, mean, median", "mean=62.5, median=65.0")
    check(s["pass_rate"] == 0.75 and s["best_programme"] == "SE", "pass_rate 0.75 and best programme SE", "SE mean 65 beats CS mean 60")
    full = summarise(ra.make_records())
    check(full["n"] == 30 and 25 <= full["mean"] <= 99, "works on the full generated table", "do not hard-code numbers")
except Exception as err:               # an unfinished TODO can cause an error: that is normal
    print('❌ Not yet: your code raised', type(err).__name__, '-', err, '(finish the TODO above first)')

### 🛠️ Task 3: Mock practical: fix the seeded bugs

The function `top_students(df, n)` should return the `n` students with the **highest** scores (highest first). The function `programme_counts(df)` should return a dictionary `{programme: number_of_students}`. Each has **one** bug. Find and fix them, then in `bug_report` explain each bug in one sentence (what was wrong, how you found it).

In [ ]:
def top_students(df, n):
    return df.sort_values("score").head(n)                   # BUG somewhere in this line

def programme_counts(df):
    counts = {}
    for programme in df["programme"]:
        counts[programme] = 1                                # BUG somewhere in this line
    return counts

bug_report = []   # TODO: two strings, one per bug

probe = pd.DataFrame({"student_id": [1, 2, 3], "programme": ["CS", "CS", "SE"], "score": [50, 90, 70]})
print(top_students(probe, 2)["score"].tolist(), programme_counts(probe))

In [ ]:
# ✅ CHECK YOUR WORK: run this cell after you finish the task
try:                                   # run the checks safely
    probe = pd.DataFrame({"student_id": [1, 2, 3], "programme": ["CS", "CS", "SE"], "score": [50, 90, 70]})
    check(top_students(probe, 2)["score"].tolist() == [90, 70], "top_students returns the highest scores, highest first", "sort_values(..., ascending=False)")
    check(top_students(probe, 10)["score"].tolist() == [90, 70, 50], "n larger than the table is handled", "head(n) already handles this")
    check(programme_counts(probe) == {"CS": 2, "SE": 1}, "programme_counts counts correctly", "increase the existing count with counts.get(key, 0) + 1")
    check(isinstance(bug_report, list) and len(bug_report) == 2 and all(len(b.split()) >= 8 for b in bug_report), "two bug explanations written", "one sentence per bug: what was wrong and how you found it")
except Exception as err:               # an unfinished TODO can cause an error: that is normal
    print('❌ Not yet: your code raised', type(err).__name__, '-', err, '(finish the TODO above first)')

### 🛠️ Task 4: Write the README

Assign to `readme` a Markdown string for **your own** version of the project. It must contain the headings `# `, `## Purpose`, `## Installation`, `## Usage`, `## Testing`, `## Limitations` and `## Author` (your name, matric number, department).

In [ ]:
readme = """
# TODO project title

## Purpose
TODO

"""
print(readme)

In [ ]:
# ✅ CHECK YOUR WORK: run this cell after you finish the task
try:                                   # run the checks safely
    needed = ["# ", "## Purpose", "## Installation", "## Usage", "## Testing", "## Limitations", "## Author"]
    missing = [h for h in needed if h not in readme]
    check(not missing, "README has all required headings", f"missing: {missing}")
    check(len(readme.split()) >= 60 and "TODO" not in readme, "README is complete (60+ words, no TODO)", "replace every TODO with real content")
except Exception as err:               # an unfinished TODO can cause an error: that is normal
    print('❌ Not yet: your code raised', type(err).__name__, '-', err, '(finish the TODO above first)')

### 🛠️ Task 5: Plan a 5-minute demonstration

Create `demo_plan`: a list of **at least 5** `(step, minutes)` tuples (for example `("Show the problem and the data", 1)`). The total must be **at most 5 minutes** and must include a step that mentions `test` and a step that mentions `limitation`.

In [ ]:
demo_plan = [
    # ("Introduce the problem", 1),
]
print(sum(m for _, m in demo_plan))

In [ ]:
# ✅ CHECK YOUR WORK: run this cell after you finish the task
try:                                   # run the checks safely
    text = " ".join(s.lower() for s, _ in demo_plan)
    check(len(demo_plan) >= 5 and all(isinstance(t, tuple) and len(t) == 2 for t in demo_plan), "at least five (step, minutes) tuples", "write tuples like (\"Run the tests\", 1)")
    check(0 < sum(m for _, m in demo_plan) <= 5, "total time is at most 5 minutes", "adjust the minutes")
    check("test" in text and "limitation" in text, "includes testing and limitations", "mention both in step descriptions")
except Exception as err:               # an unfinished TODO can cause an error: that is normal
    print('❌ Not yet: your code raised', type(err).__name__, '-', err, '(finish the TODO above first)')

### 🛠️ Task 6: Reflection (honest and specific)

Assign to `reflection` a dictionary with keys `strongest_skill`, `hardest_concept`, `next_step` and `evidence`. Each value must be a full sentence (at least 8 words). `evidence` must point to **something you actually produced** (a task, test, or bug you fixed).

In [ ]:
reflection = {
    "strongest_skill": "",
    "hardest_concept": "",
    "next_step": "",
    "evidence": "",
}

In [ ]:
# ✅ CHECK YOUR WORK: run this cell after you finish the task
try:                                   # run the checks safely
    keys = {"strongest_skill", "hardest_concept", "next_step", "evidence"}
    check(set(reflection) == keys, "four keys present", "use exactly these key names")
    check(all(len(str(v).split()) >= 8 for v in reflection.values()), "each answer is a full sentence", "write at least 8 words per answer")
except Exception as err:               # an unfinished TODO can cause an error: that is normal
    print('❌ Not yet: your code raised', type(err).__name__, '-', err, '(finish the TODO above first)')

### ❓ Quick check: answer in your own words first

**Q1. Why does a README matter if the code already works?**

> _Your answer:_ 

**Q2. A test passes on correct code. How do you know the test is *good*?**

> _Your answer:_ 

**Q3. Where do bugs most often hide in selection and loops?**

> _Your answer:_ 

**Q4. Why use `seed=42` in an assessed project?**

> _Your answer:_ 

## 📝 Take-home assignment

**Final Project (individual).** Extend the Student Records Analyser into your own mini-application of about 150 lines. You may keep the student-records theme or choose a different **fictional or openly licensed** dataset.

Submit in your GitHub repository:
1. `app.py` (or a module) with at least **5 functions**, every line explained in comments
2. `test_app.py` with at least **8 tests** (normal, boundary, invalid)
3. `README.md` with all required headings
4. One saved chart (`.png`) with title and labelled axes
5. This notebook with all tasks completed

In the cell below, record your submission details and a self-assessment against the marking guide.

In [ ]:
submission = {
    "repository_url": "",         # TODO: link to your GitHub repository
    "function_count": 0,          # TODO: number of functions in your app
    "test_count": 0,              # TODO: number of tests
    "uses_sqlite": False,         # TODO: True / False
    "ai_or_help_declared": "",    # TODO: who/what helped you? Write "none" if no help
}
print(submission)

In [ ]:
# ✅ CHECK YOUR WORK
try:                                   # run the checks safely
    check(str(submission["repository_url"]).startswith("https://github.com/"), "repository link looks valid", "paste your GitHub repository URL")
    check(isinstance(submission["function_count"], int) and submission["function_count"] >= 5, "at least 5 functions", "add functions")
    check(isinstance(submission["test_count"], int) and submission["test_count"] >= 8, "at least 8 tests", "add normal, boundary and invalid tests")
    check(isinstance(submission["uses_sqlite"], bool) and len(str(submission["ai_or_help_declared"]).split()) >= 1, "SQLite flag and help declaration completed", "declare any help honestly; write none if none")
except Exception as err:               # an unfinished TODO can cause an error: that is normal
    print('❌ Not yet: your code raised', type(err).__name__, '-', err, '(finish the TODO above first)')

---
## 🪞 Reflection (write 3 to 5 sentences)
1. What is the single most important idea you learned this week?
2. Which task was hardest, and how did you get unstuck?
3. What will you do differently in Week 12?

*(Double-click this cell and type your answers below.)*

**My reflection:** _write here_

---
## ✔️ Submission checklist
- [ ] Every `TODO` replaced and every ✅ check passes
- [ ] Notebook runs from top to bottom with **Runtime → Restart and run all** and no errors
- [ ] Each task has a short comment explaining *why* your solution works
- [ ] Reflection written
- [ ] Declaration below completed
- [ ] Saved to GitHub: **File → Save a copy in GitHub** (or downloaded as `.ipynb` and uploaded)

## 📜 Declaration of sources and AI assistance
I declare that this work is my own, except for the following:

- **Collaboration (names):** _none / write here_
- **External code or websites used:** _none / write here_
- **AI assistance used (tool, prompt, which part):** _none / write here_
- **I can explain every line I submitted:** _yes / no_

**Student name:** _________  **Matric no.:** _________  **Date:** _________

---
*OOU-CSC 333 · Computer Programming III (Laboratory) · Dr Sakinat Folorunso, Associate Professor of AI Systems, Department of Computer Science, Olabisi Onabanjo University, Ago-Iwoye, Nigeria*
